# LLM Architecture and Embeddings

This notebook builds the conceptual bridge from classical neural networks to Large Language Models.

In previous weeks, our models usually received numeric features directly: areas, prices, pixel values, or engineered variables. LLMs are still neural networks, but their input begins as text. Text is symbolic, variable-length, and sequential, so we need a pipeline that turns sentences into tensors.

This notebook is inspired by the early text-processing chapters of Sebastian Raschka's *Build a Large Language Model From Scratch*, especially the path from raw text to tokens, token IDs, token embeddings, positional embeddings, and training windows. The examples here are rewritten as a bootcamp teaching notebook: small examples first, visible tensor shapes, and checkpoints after each idea.

The central question is:

> How does raw text become something a neural network can process?

By the end, the following pipeline should feel concrete:

```text
raw text -> tokens -> token IDs -> embeddings + positions -> Transformer input
```


## Learning Objectives

By the end of this notebook you should be able to:

- Explain why text needs preprocessing before it can enter a neural network.
- Describe the path from text to tokens, token IDs, embeddings, and model inputs.
- Build a simple tokenizer and vocabulary.
- Explain why token IDs are addresses, not numeric measurements.
- Use special tokens such as `<unk>` and `<eot>`.
- Explain why modern tokenizers often use subwords.
- Create input-target pairs for next-token prediction.
- Explain token embeddings and positional embeddings.
- Identify which parts are fixed preprocessing decisions and which parts are learned parameters.
- Describe the high-level architecture of a decoder-only LLM.

As you read, keep track of the **shape** of the data. LLM engineering is full of tensors, and many bugs are really shape misunderstandings.


## 0. Setup

This notebook uses the Python standard library by default. `numpy` and `matplotlib` are helpful but not essential. `torch` is optional and only used to show the production-style embedding layer.

We intentionally avoid external tokenizer libraries in the first sections. Production tokenizers are more sophisticated, but the simple implementation makes the ideas visible.


In [ ]:
import math
import re
import random
from collections import Counter

try:
    import numpy as np
except ImportError:
    np = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

print("Setup complete")
print("numpy available:", np is not None)
print("matplotlib available:", plt is not None)


## 1. The Big Picture

A decoder-only LLM can be introduced with this pipeline:

```text
raw text -> tokens -> token IDs -> token embeddings + position embeddings
         -> Transformer blocks -> next-token logits -> probability distribution
```

The model does not receive characters or words directly. It receives arrays of numbers.

The first half of the pipeline is about constructing those arrays:

| Stage | Purpose |
| --- | --- |
| Tokenization | Split raw text into units. |
| Vocabulary | Map each token to an integer ID. |
| Embeddings | Map token IDs to learned vectors. |
| Positional information | Tell the model where each token appears. |
| Training windows | Create input-target pairs for next-token prediction. |

A useful mental distinction:

- **Tokenization and vocabulary** are usually fixed before training.
- **Embeddings and Transformer weights** are learned during training.
- **Next-token probabilities** are generated by the model during inference.


### Concept Note: LLM Input Pipeline

Token IDs are not measurements. They are indices.

The ID `17` is not "larger" than the ID `3` in a meaningful semantic way. It is just an address that points to a row in an embedding table.

The learned meaning enters through the embedding table and the rest of the model parameters.


## 2. A Tiny Course Corpus

To make every step easy to inspect, we will use a tiny corpus about the bootcamp itself.

A **corpus** is a collection of text used for training or analysis. Real LLMs are trained on enormous corpora. Here we use only a few sentences so we can inspect every token and every ID.

Later you can replace this with a longer text file such as `wsembeddings/the-verdict.txt`.


In [ ]:
corpus = """
Machine learning models learn patterns from data.
Neural networks learn useful representations from examples.
Large language models learn to predict the next token.
Embeddings turn tokens into vectors.
Retrieval augmented generation uses embeddings to find relevant context.
""".strip()

print(corpus)


### Reading the Corpus

The corpus is deliberately small and repetitive. That makes it easier to see repeated tokens such as `learn`, `models`, and `embeddings`.

In a real pretraining corpus, the model would see billions or trillions of tokens. Here we only need enough text to demonstrate the pipeline.


## 3. Tokenization

Tokenization chooses the units of text that the model will process.

A token can be:

- A word.
- A punctuation mark.
- A subword piece.
- A byte or character-like unit.
- A special marker such as an end-of-text token.

The simple tokenizer below uses a regular expression to split English words, numbers, and punctuation. This is not how modern production tokenizers work, but it exposes the core idea without hiding it behind a library.

Tokenization is a modeling decision. Different tokenizers can split the same text differently, and those differences affect vocabulary size, sequence length, multilingual behavior, and code handling.


In [ ]:
def simple_tokenize(text):
    pattern = r"[A-Za-z]+|\d+|[^\w\s]"
    return re.findall(pattern, text.lower())

tokens = simple_tokenize(corpus)
print(tokens)
print("Number of tokens:", len(tokens))
print("Unique tokens:", len(set(tokens)))


### Reading the Tokenization Output

The output is a list of lowercase tokens. Punctuation such as `.` appears as its own token.

The counts help us see the difference between total tokens and unique tokens:

- Total tokens: how long the text sequence is.
- Unique tokens: how many distinct vocabulary entries appear.

A repeated word such as `learn` contributes multiple total tokens but only one unique token.


### Checkpoint

Consider the sentence `Large language models learn patterns.`

- What are the tokens?
- Are punctuation marks tokens?
- What information is lost by lowercasing?
- What would happen with Spanish accents, emojis, or code snippets?
- Would `model`, `models`, and `modeling` be treated as related by this tokenizer?

Important limitation: this simple tokenizer is good for teaching, not for production. Modern LLM tokenizers handle much more variety: uppercase/lowercase, accents, whitespace, code, names, numbers, punctuation, and multilingual text.


## 4. Vocabulary and Token IDs

A vocabulary maps each token to an integer.

```text
token -> integer ID
```

The integer is only an address. It does not mean that token `7` is larger, better, or more important than token `3`.

This is different from numeric features in classical machine learning. In linear regression, a house size of `200` really is larger than a house size of `100`. But a token ID of `200` is not semantically larger than a token ID of `100`.

The model needs token IDs because embedding tables are indexed by integers. The ID chooses a row; the row contains the learned vector.


In [ ]:
special_tokens = ["<unk>", "<eot>"]
regular_tokens = sorted(set(tokens))
vocab = {tok: idx for idx, tok in enumerate(special_tokens + regular_tokens)}
id_to_token = {idx: tok for tok, idx in vocab.items()}

print("Vocabulary size:", len(vocab))
for tok, idx in list(vocab.items())[:15]:
    print(f"{tok:>14} -> {idx}")


### Reading the Vocabulary Output

The vocabulary begins with special tokens and then includes the unique tokens found in the corpus.

The mapping is deterministic here because we sort the regular tokens. In production, vocabulary construction depends on the tokenizer training algorithm.

The exact integer assigned to each token is not meaningful. What matters is consistency: the same token must always map to the same ID for a given tokenizer and model.


In [ ]:
def encode(text, vocab):
    return [vocab.get(tok, vocab["<unk>"]) for tok in simple_tokenize(text)]

def decode(token_ids, id_to_token):
    return " ".join(id_to_token[i] for i in token_ids)

sample = "Language models use embeddings."
encoded = encode(sample, vocab)
print("Text:", sample)
print("Token IDs:", encoded)
print("Decoded:", decode(encoded, id_to_token))


### Reading the Encoding Output

The sample sentence includes the word `use`, but our vocabulary contains `uses`, not `use`. Because this tiny tokenizer uses exact word matching, `use` becomes `<unk>`.

This demonstrates a real design problem:

- Small word-level vocabularies create many unknown tokens.
- Large word-level vocabularies become inefficient.
- Subword tokenizers provide a practical compromise.

The decoded text is not identical to the original sentence because tokenization, lowercasing, and unknown-token replacement lose information.


### Concept Note: Token IDs Are Addresses

The token ID sequence is a compact symbolic representation. The model still needs a numeric vector representation, which comes next.

A useful analogy:

- Token ID: a library shelf address.
- Embedding vector: the actual book content the model can read.

The address alone is not meaningful. It becomes useful when it retrieves the vector from the embedding table.


## 5. Special Tokens

Special tokens mark events that are not ordinary words.

In this tiny notebook we use:

- `<unk>` for unknown tokens.
- `<eot>` for end of text.

Why do we need them?

- `<unk>` gives the tokenizer a fallback when it sees a token outside the vocabulary.
- `<eot>` tells the model that one text ended and another text may begin.

In chat models you may also see markers for system, user, assistant, tool calls, padding, or document boundaries. Those markers are part of how a general language model is turned into a conversational assistant.


In [ ]:
texts = [
    "Embeddings represent tokens.",
    "Agents can use tools."
]

stream = []
for text in texts:
    stream.extend(encode(text, vocab))
    stream.append(vocab["<eot>"])

print("Stream IDs:", stream)
print("Stream tokens:", [id_to_token[i] for i in stream])


### Reading the Special Token Stream

The stream combines multiple texts into one sequence and inserts `<eot>` after each text.

Notice how unknown words become `<unk>` because our tiny vocabulary was built from the original corpus only. This is a limitation of small word-level vocabularies.

The `<eot>` marker gives the model a boundary signal. Without such boundaries, examples from different documents or conversations can blur together.


## 6. Why Modern Tokenizers Use Subwords

A word-level tokenizer has a problem: unseen words become `<unk>`.

For example, if our vocabulary contains `agent` but not `agentic`, a word-level tokenizer may lose the word entirely and replace it with `<unk>`. That is bad because the model receives less information.

Modern LLMs usually use subword tokenizers, often based on Byte Pair Encoding or related algorithms. Subword tokenization lets the model represent rare names, new terms, code fragments, and multilingual text by composing smaller pieces.

Example intuition:

```text
learning -> learn + ing
agentic  -> agent + ic
```

The toy function below is not real BPE. It is only a small demonstration of the idea: represent unfamiliar words using reusable known pieces.


In [ ]:
def toy_subword_tokenize(word, known_pieces):
    """Greedy longest-piece tokenizer for demonstration only."""
    word = word.lower()
    pieces = []
    i = 0
    while i < len(word):
        match = None
        for j in range(len(word), i, -1):
            piece = word[i:j]
            if piece in known_pieces:
                match = piece
                break
        if match is None:
            pieces.append(word[i])
            i += 1
        else:
            pieces.append(match)
            i += len(match)
    return pieces

known_pieces = {"agent", "retrieval", "embedding", "embeddings", "learn", "ing", "s", "model", "language", "rag"}
for word in ["embeddings", "learning", "retrieval", "agentic"]:
    print(f"{word:>10} -> {toy_subword_tokenize(word, known_pieces)}")


### Reading the Subword Output

The toy tokenizer tries to find known pieces inside each word.

When it sees `learning`, it can use `learn` and `ing`. When it sees a word such as `agentic`, it may still fall back to individual characters for pieces it does not know.

Real BPE tokenizers learn common pieces from a large corpus. Frequent patterns become reusable tokens. Rare words can still be represented as combinations of smaller pieces.


## 7. Next-Token Prediction

LLMs are commonly pretrained with a next-token prediction objective.

Given a sequence, the model learns to predict the following token at every position.

If the input is:

```text
Machine learning models learn
```

The target is shifted by one token:

```text
learning models learn patterns
```

This shift is the core training signal. The model sees many partial sequences and learns to assign high probability to the actual next token.

This objective is powerful because text provides its own labels. We do not need a human to label every sentence. The next token is already present in the corpus.


In [ ]:
all_ids = encode(corpus, vocab) + [vocab["<eot>"]]

context_length = 6
for start in range(0, 4):
    x = all_ids[start:start + context_length]
    y = all_ids[start + 1:start + context_length + 1]
    print("input :", [id_to_token[i] for i in x])
    print("target:", [id_to_token[i] for i in y])
    print()


### Reading the Shifted Examples

Each printed pair shows the same text stream from two perspectives:

- `input`: what the model receives.
- `target`: what the model should predict next.

The target sequence is not a separate human label. It is created by shifting the original token sequence by one position.

This is one reason language modeling scales: raw text can be converted into many supervised training examples automatically.


## 8. Data Sampling With a Sliding Window

This is the explicit data-sampling step that turns one long token stream into many training examples.

An LLM is trained to predict the next token. But a corpus is usually just a long sequence of token IDs:

```text
[t0, t1, t2, t3, t4, t5, t6, ...]
```

The sliding-window procedure repeatedly takes a fixed-length slice from that stream:

```text
input  = [t0, t1, t2, t3]
target = [t1, t2, t3, t4]
```

Then it slides forward and creates another pair:

```text
input  = [t1, t2, t3, t4]
target = [t2, t3, t4, t5]
```

The **input** is what the model sees. The **target** is the same sequence shifted one token to the right. This means each position teaches the model what token should come next.

Two parameters control the sampling:

- `max_length` or `context_length`: how many tokens go into each input sequence.
- `stride`: how far the window moves between examples.

A small stride creates many overlapping examples. A larger stride creates fewer, less-overlapping examples.


### 8.1 One Sequence, Many Prediction Tasks

Before we build a dataset function, look at a single context window.

For a context length of 4, the model can be trained on progressively longer contexts:

```text
[t0]             -> t1
[t0, t1]         -> t2
[t0, t1, t2]     -> t3
[t0, t1, t2, t3] -> t4
```

This is why next-token prediction creates such a rich training signal from plain text.


In [ ]:
demo_context_size = 4
demo_sample = all_ids[:demo_context_size + 1]

print("Token IDs:", demo_sample)
print("Tokens:", [id_to_token[i] for i in demo_sample])
print()

for i in range(1, demo_context_size + 1):
    context = demo_sample[:i]
    desired = demo_sample[i]
    print([id_to_token[token_id] for token_id in context], "---->", id_to_token[desired])


### Reading the Progressive Prediction Output

Each line is one training signal.

The model receives the tokens on the left and should predict the token on the right. During real training, the model does this for many positions, many windows, many batches, and many documents.

The important idea is that the corpus provides the labels automatically. We do not manually label the next token; we create the label by shifting the sequence.


### 8.2 A Small Dataset Function

The function below implements sliding-window sampling for the full token stream.

It returns two lists:

- `inputs`: chunks the model receives.
- `targets`: the same chunks shifted one token to the right.

In production code, this logic is usually wrapped in a dataset class and a dataloader. A dataloader groups examples into batches, shuffles examples during training, and sends tensors to the accelerator. Here we keep the logic transparent.


In [ ]:
def make_windows(token_ids, context_length=6, stride=3):
    inputs = []
    targets = []
    for start in range(0, len(token_ids) - context_length, stride):
        inputs.append(token_ids[start:start + context_length])
        targets.append(token_ids[start + 1:start + context_length + 1])
    return inputs, targets

inputs, targets = make_windows(all_ids, context_length=6, stride=3)
print("Number of examples:", len(inputs))
print("First input IDs:", inputs[0])
print("First target IDs:", targets[0])
print("First input tokens:", [id_to_token[i] for i in inputs[0]])
print("First target tokens:", [id_to_token[i] for i in targets[0]])


### Reading the Window Output

The first input and target have the same length. The target is the input shifted one position to the right.

This is what lets the model learn many predictions from one sequence. For a context of length 6, the model can learn to predict token 2 from token 1, token 3 from tokens 1-2, and so on, depending on how the training loss is applied.

The dataset function returns lists now, but a production training pipeline would return tensors and batches.


### 8.3 From Windows to Batches

Training usually processes several windows at once. A group of examples is called a **batch**.

If we have:

```text
batch size = 2
context length = 6
```

then the batch of token IDs has shape:

```text
2 x 6
```

After embedding lookup, that becomes:

```text
2 x 6 x embedding_dimension
```

The helper below creates simple batches from our input-target windows. It is a lightweight version of what a PyTorch `DataLoader` would do.


In [ ]:
def make_batches(inputs, targets, batch_size=2, drop_last=False):
    batches = []
    for start in range(0, len(inputs), batch_size):
        input_batch = inputs[start:start + batch_size]
        target_batch = targets[start:start + batch_size]
        if drop_last and len(input_batch) < batch_size:
            continue
        batches.append((input_batch, target_batch))
    return batches

batches = make_batches(inputs, targets, batch_size=2, drop_last=True)
first_input_batch, first_target_batch = batches[0]

print("Number of full batches:", len(batches))
print("Input batch shape:", (len(first_input_batch), len(first_input_batch[0])))
print("Target batch shape:", (len(first_target_batch), len(first_target_batch[0])))
print()
print("First input batch as tokens:")
for row in first_input_batch:
    print([id_to_token[i] for i in row])
print()
print("First target batch as tokens:")
for row in first_target_batch:
    print([id_to_token[i] for i in row])


### Reading the Batch Output

The batch groups multiple windows together.

The model will eventually receive batches because training one example at a time is inefficient. Batching lets hardware process many examples in parallel.

The key shape progression is:

```text
token IDs:      batch_size x context_length
embeddings:     batch_size x context_length x embedding_dimension
model outputs:  batch_size x context_length x vocabulary_size
```

That final dimension, `vocabulary_size`, appears because the model predicts a probability distribution over possible next tokens at each position.


### Mini Lab 1

Experiment with the sliding-window parameters.

Change `context_length`, `stride`, and `batch_size` above.

Observe:

- What happens to the number of examples?
- What happens when stride is smaller than context length?
- What happens when stride equals context length?
- What happens to the number of batches when `batch_size` changes?
- Why might overlapping windows be useful during training?
- Why might too much overlap be inefficient or redundant?

Then try to answer this conceptual question:

> If the model predicts the next token at every position, why do we need both `inputs` and `targets`?


## 9. Token Embeddings

An embedding layer is a lookup table with shape:

```text
vocabulary_size x embedding_dimension
```

Each token ID selects one row.

If the vocabulary has 34 tokens and the embedding dimension is 4, the embedding table has shape:

```text
34 x 4
```

In real LLMs, the embedding dimension is much larger: hundreds, thousands, or more. The purpose is the same. Each token becomes a vector that the neural network can process.

During training, the model adjusts these rows so they become useful for prediction. The vectors are not manually designed. They are learned from the training objective.


In [ ]:
random.seed(7)
vocab_size = len(vocab)
embedding_dim = 4

def random_vector(dim, scale=0.02):
    return [random.gauss(0.0, scale) for _ in range(dim)]

embedding_table = [random_vector(embedding_dim) for _ in range(vocab_size)]
batch_ids = inputs[:2]
token_embeddings = [
    [embedding_table[token_id] for token_id in example]
    for example in batch_ids
]

print("Embedding table shape:", (len(embedding_table), len(embedding_table[0])))
print("Batch token ID shape:", (len(batch_ids), len(batch_ids[0])))
print("Batch embedding shape:", (len(token_embeddings), len(token_embeddings[0]), len(token_embeddings[0][0])))
print("First token:", id_to_token[batch_ids[0][0]])
print("First token embedding:", token_embeddings[0][0])


### Reading the Embedding Output

The printed shapes are the key result.

```text
Embedding table shape: vocabulary size x embedding dimension
Batch token ID shape: batch size x context length
Batch embedding shape: batch size x context length x embedding dimension
```

This is the first moment where the text pipeline has become a neural-network input tensor.

The example uses random vectors because we are not training a model in this notebook. In a real LLM, these vectors would be updated during training by gradient descent.


### Concept Note: Embeddings Are Learned Lookup Tables

Embeddings are not manually designed semantic coordinates.

At initialization, they may be random. They become useful because the training objective repeatedly adjusts them.

The model learns vectors that help solve the next-token prediction task. Semantic structure can emerge from that training process, but it is not hand-coded into the token IDs.


## 10. Positional Embeddings

If we only use token embeddings, the model sees token identity but not position.

That is a problem because order matters:

```text
dog bites man
man bites dog
```

These contain the same words, but the meaning changes because the order changes.

Decoder-only LLMs usually combine token representations with position information before the Transformer blocks. One simple approach is to add a learned positional vector to each token vector.

First separate the two ideas:

- A **token vector** represents token identity: *what token is this?*
- A **positional vector** represents a slot in the context window: *where is this token?*

A positional vector is selected from a position embedding table. Position `0` has one learned vector, position `1` has another learned vector, position `2` has another learned vector, and so on up to the maximum context length.

For example, in a context window of length 6:

```text
token slot:      0        1        2        3        4        5
token vector:    v_tok0   v_tok1   v_tok2   v_tok3   v_tok4   v_tok5
position vector: v_pos0   v_pos1   v_pos2   v_pos3   v_pos4   v_pos5
```

The same token can receive a different final representation depending on where it appears, because it is added to a different positional vector.

Conceptually:

```text
token embedding + position embedding = input representation
```

This gives the model both pieces of information: what the token is and where it appears.


In [ ]:
max_context_length = context_length
position_table = [random_vector(embedding_dim) for _ in range(max_context_length)]
positions = list(range(context_length))
position_embeddings = [position_table[pos] for pos in positions]

model_inputs = []
for example_embeddings in token_embeddings:
    combined_example = []
    for token_vector, position_vector in zip(example_embeddings, position_embeddings):
        combined_example.append([
            token_value + position_value
            for token_value, position_value in zip(token_vector, position_vector)
        ])
    model_inputs.append(combined_example)

print("Token embeddings shape:", (len(token_embeddings), len(token_embeddings[0]), len(token_embeddings[0][0])))
print("Position embeddings shape:", (len(position_embeddings), len(position_embeddings[0])))
print("Combined model input shape:", (len(model_inputs), len(model_inputs[0]), len(model_inputs[0][0])))


### Reading the Positional Embedding Output

The output shapes show three levels:

```text
batch size x context length x embedding dimension
```

The token embeddings and position embeddings have compatible dimensions so they can be added.

Notice that the same position vector is reused for every example in the batch. Position `0` gets the position-0 vector, position `1` gets the position-1 vector, and so on.

After addition, each token representation contains both identity information and position information.


## 11. Optional: The Same Idea in PyTorch

Production LLM code typically uses `torch.nn.Embedding`.

The PyTorch version does the same conceptual operation as our manual list lookup:

```text
integer token IDs -> rows from an embedding matrix
```

This section is optional. If `torch` is unavailable, the conceptual explanation above is enough. The point is not the library; the point is the shape transformation.


In [ ]:
try:
    import torch

    torch.manual_seed(7)
    embedding_layer = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=4)
    batch_tensor = torch.tensor(inputs[:2], dtype=torch.long)
    embedded = embedding_layer(batch_tensor)

    print("Batch tensor shape:", tuple(batch_tensor.shape))
    print("Embedded batch shape:", tuple(embedded.shape))
except ImportError:
    print("PyTorch is not installed. Skipping optional section.")


### Reading the PyTorch Output

If PyTorch is installed, the output shape should match the manual embedding example:

```text
batch size x context length x embedding dimension
```

The important idea is that `torch.nn.Embedding` expects integer IDs, not one-hot vectors and not raw strings. It returns vectors that can be passed into later neural-network layers.


## 12. High-Level Decoder-Only LLM Architecture

Now we can place the preprocessing pipeline inside the full model.

```text
input token IDs
    -> token embedding lookup
    -> position embedding lookup
    -> repeated Transformer blocks
         -> self-attention
         -> feed-forward network
         -> residual connections and normalization
    -> output projection
    -> next-token logits
```

A **decoder-only** model predicts future tokens from previous tokens. During generation, it repeatedly:

1. Takes the current context.
2. Produces a probability distribution over the vocabulary.
3. Selects or samples the next token.
4. Appends that token to the context.
5. Repeats.

This notebook does not implement a Transformer from scratch. The purpose here is to understand what the model receives and what it predicts.

The next conceptual step would be self-attention: how each token representation gets updated using information from previous tokens in the context window.


### Checkpoint: Fixed or Learned?

Classify each item as fixed preprocessing, learned parameter, intermediate representation, or generated output:

| Item | Category |
| --- | --- |
| Regex tokenizer rule | ? |
| Vocabulary mapping | ? |
| Token IDs for a sentence | ? |
| Token embedding matrix | ? |
| Position embedding matrix | ? |
| Combined model input tensor | ? |
| Next-token probability distribution | ? |

Suggested answer:

| Item | Category |
| --- | --- |
| Regex tokenizer rule | Fixed preprocessing decision |
| Vocabulary mapping | Fixed preprocessing decision |
| Token IDs for a sentence | Intermediate representation |
| Token embedding matrix | Learned parameter |
| Position embedding matrix | Learned parameter |
| Combined model input tensor | Intermediate representation |
| Next-token probability distribution | Generated output |


## 13. Summary

Key takeaways:

- LLMs process token sequences, not raw text.
- Tokenization defines the units the model sees.
- A vocabulary maps tokens to token IDs.
- Token IDs are symbolic addresses into embedding tables.
- Special tokens mark boundaries and non-ordinary text events.
- Subword tokenization helps avoid excessive unknown tokens.
- Next-token prediction uses shifted input-target sequences.
- Data sampling with a sliding window turns a long token stream into many training examples.
- Batching groups multiple windows so training can run efficiently.
- Embeddings are learned vectors used as model inputs.
- Positional information tells the model where tokens appear.
- The combined token and position representations enter Transformer blocks.

Final mental model:

```text
Text becomes tokens.
Tokens become IDs.
Sliding windows create input-target examples.
IDs select embedding vectors.
Position vectors add order information.
Transformer blocks update the sequence.
The model predicts the next token.
```

These ideas become the foundation for RAG and agentic architectures, where embeddings are used not only inside the model, but also as retrieval infrastructure.
